# Feasibility study: rooftop PV for a business centre in Bishkek (Huawei, 3 options)

The notebook runs `model.py`: an hourly generation / load / battery balance and financial metrics (NPV, IRR, payback, ROI, PI, LCOE). All assumptions are in the `A` and `P` dictionaries in `model.py`.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import model

res = model.run()
summary = pd.DataFrame({k: {
    'Option': r['variant']['name'], 'kWp': r['kwp'], 'CAPEX, $': r['capex_usd'], '$/kWp': r['usd_per_kwp'],
    'Generation, kWh': r['energy']['pv'], 'Self-consumption, kWh': r['energy']['self_used'],
    'NPV, KGS M': round(r['fin']['npv_som']/1e6, 2), 'IRR, %': round(r['fin']['irr']*100, 1),
    'Payback, years': r['fin']['payback'] and round(r['fin']['payback'], 1),
    'PI': round(r['fin']['pi'], 2), 'LCOE, KGS/kWh': round(r['fin']['lcoe_gen'], 2)} for k, r in res.items()}).T
summary

## Monthly balance, option 1

In [ ]:
df = model.build_year()
load = df.groupby('month')['load'].sum()
pv = pd.Series(res['V1']['energy']['monthly_pv'], index=range(1, 13))
ax = pd.DataFrame({'Consumption': load, 'PV generation': pv}).plot.bar(figsize=(10, 4), color=['#bbb', '#2b7bd6'])
ax.set_ylabel('kWh'); ax.set_xlabel('month'); plt.tight_layout()

## Cumulative cash flow

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k, r in res.items():
    cf = pd.Series(r['fin']['cf']).cumsum() / 1e6
    ax.plot(range(len(cf)), cf, label=f"{k}: {r['variant']['name']}")
ax.axhline(0, color='grey', lw=0.8); ax.set_xlabel('year'); ax.set_ylabel('KGS M, nominal'); ax.legend(); plt.tight_layout()

## Sensitivity

One factor changes at a time, the others stay at base. Running `python model.py` prints the full table.

In [ ]:
cases = {'Base': ({}, {}), 'No tariff growth': ({'esc_to_2030': 0, 'esc_after': 0}, {}), 'Discount rate 10%': ({'discount': 0.10}, {}), 'Discount rate 18%': ({'discount': 0.18}, {}), 'Buy-back KGS 3.5': ({'export_som': 3.5}, {})}
pd.DataFrame({name: {k: round(r['fin']['npv_som']/1e6, 2) for k, r in model.run(a, p).items()} for name, (a, p) in cases.items()}).T.add_suffix(' NPV, KGS M')